# Strands Agents with LaunchDarkly AI Configs

[LaunchDarkly AI Configs](https://launchdarkly.com/docs/home/ai-configs) externalize the model, prompt, parameters, and tools your agent uses, so you can change them in the LaunchDarkly UI without redeploying.

The [Strands Agents SDK](https://strandsagents.com) builds agents with pluggable model classes (`OpenAIModel`, `AnthropicModel`, `BedrockModel`). LaunchDarkly picks *which* variation to serve at runtime; Strands just reads it back and instantiates the matching model class. The result: swap `gpt-5` for `claude-sonnet-4-6` (or Bedrock-hosted Claude) without changing a line of agent code.

This sample creates an agent-mode AI Config with three variations (OpenAI, Anthropic, Bedrock), attaches a governed `get_order_status` tool, and runs three turns through a Strands `Agent` with `SlidingWindowConversationManager` for multi-turn memory. The final section builds an Agent Graph that wires a deep-reasoning specialist node onto the root agent.

## What you'll learn
- Map a LaunchDarkly AI Config variation to a Strands model class (`create_strands_model`)
- Drive an agent's tool list from LaunchDarkly rather than hardcoded Python lists
- Track duration, tokens, success/error, and tool-call counts per turn with the LaunchDarkly AI SDK
- Compose multiple agents into an Agent Graph and read the topology back via the SDK

## Prerequisites
- Python 3.10+
- A LaunchDarkly account with an API token (Writer role). The notebook creates the project, AI Config, variations, and tool for you, and fetches the SDK key from the API at runtime.
- `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` (Strands reads them from the environment)
- Optional: AWS credentials with Bedrock model access for the `bedrock-claude-agent` variation
- A `.env` file in this directory with the keys above

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade launchdarkly-server-sdk launchdarkly-server-sdk-ai strands-agents strands-agents-tools boto3 python-dotenv requests

In [ ]:
import os
import json
import time
import requests
from dotenv import load_dotenv

# Load .env from the directory containing this notebook.
load_dotenv()

LD_API_TOKEN = os.environ["LAUNCHDARKLY_API_TOKEN"]
BASE_URL     = "https://app.launchdarkly.com/api/v2"
# Project + environment are overridable so this sample can run against any
# LaunchDarkly account. Defaults create a fresh project named below.
PROJECT_KEY  = os.environ.get("LAUNCHDARKLY_PROJECT_KEY", "strands-launchdarkly-sample")
PROJECT_NAME = os.environ.get("LAUNCHDARKLY_PROJECT_NAME", "Strands + LaunchDarkly Sample")
ENVIRONMENT  = os.environ.get("LAUNCHDARKLY_ENVIRONMENT", "production")
CONFIG_KEY   = "strands-agent"
TOOL_KEY     = "get_order_status"

# `LD-API-Version: beta` is required by the agent-graphs endpoints; harmless
# on the older endpoints used elsewhere in this notebook.
HEADERS = {
    "Authorization": LD_API_TOKEN,
    "Content-Type": "application/json",
    "LD-API-Version": "beta",
}
PATCH_HEADERS = {
    "Authorization": LD_API_TOKEN,
    "Content-Type": "application/json; domain-model=launchdarkly.semanticpatch",
    "LD-API-Version": "beta",
}

# Ensure the LaunchDarkly project exists. Idempotent: 201 on first run, 409 on
# re-runs against an existing project.
r = requests.post(
    f"{BASE_URL}/projects",
    headers=HEADERS,
    json={"key": PROJECT_KEY, "name": PROJECT_NAME},
)
if r.status_code == 201:
    print(f"[OK] Project created: {PROJECT_KEY}")
elif r.status_code == 409:
    print(f"[INFO] Project already exists: {PROJECT_KEY}")
else:
    print(f"[ERROR] Project {r.status_code}: {r.text[:200]}")

# Fetch the SDK key for this environment from the API rather than reading it
# from .env. If the project was deleted and recreated, an old SDK key in .env
# won't authenticate; this always pulls the live one.
r = requests.get(
    f"{BASE_URL}/projects/{PROJECT_KEY}/environments",
    headers={"Authorization": LD_API_TOKEN},
)
SDK_KEY = next(
    (env["apiKey"] for env in r.json().get("items", []) if env["key"] == ENVIRONMENT),
    None,
)
if SDK_KEY:
    print(f"[OK] Retrieved SDK key for environment: {ENVIRONMENT}")
else:
    raise RuntimeError(f"No SDK key found for environment: {ENVIRONMENT}")

print(f"[OK] Targeting project: {PROJECT_KEY}")

## 1. Create AI Config (agent mode)

In [ ]:
# Delete the existing config so re-runs start clean. NOTE: if section 8 has run
# before, an agent graph references this config as its root and LD will refuse
# the delete with 409 ("AI Config cannot be deleted because it is used by the
# following agent graphs"). That's fine — the variation/tool/targeting cells
# below all upsert correctly against an existing config. The same goes for the
# create call: when the config still exists, LD returns 400 with the misleading
# message "could not find the 'enabled' variation" instead of 409 "already
# exists" — we treat both as the existing-config case.
requests.delete(f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}", headers=HEADERS)
time.sleep(0.5)

r = requests.post(
    f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs",
    headers=HEADERS,
    json={"key": CONFIG_KEY, "name": "Strands Agent", "mode": "agent"},
)
if r.status_code == 201:
    print(f"[OK] AI Config created: {CONFIG_KEY}")
elif r.status_code == 409 or (r.status_code == 400 and "enabled" in r.text):
    print(f"[INFO] AI Config already exists: {CONFIG_KEY} (keeping existing)")
else:
    print(f"[ERROR] {r.status_code}: {r.text[:200]}")
print(f"[INFO] https://app.launchdarkly.com/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}")

## 2. Create Variations

Three variations, each pinned to a different provider. `create_strands_model` later dispatches on `agent_config.provider.name` (with a model-id fallback for Bedrock) to pick the right Strands model class.

- `gpt-5-agent` — OpenAI gpt-5, `max_completion_tokens=1024` (gpt-5 does not accept `max_tokens` or non-default temperature)
- `claude-sonnet-agent` — Anthropic Claude Sonnet 4.6, standard parameters
- `bedrock-claude-agent` — Anthropic Claude Sonnet 4.6 *via AWS Bedrock*. Requires `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` / `AWS_REGION` (or AWS SSO) in the environment with Bedrock model access enabled.

We set `gpt-5-agent` as the default via targeting so the notebook runs deterministically on the first turn.

In [ ]:
def create_variation(body, retries=5):
    """LD occasionally returns a transient 400 right after a sibling variation is created; retry."""
    for attempt in range(retries):
        r = requests.post(
            f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}/variations",
            headers=HEADERS, json=body,
        )
        if r.status_code in (201, 409):
            return r
        time.sleep(0.6)
    return r

r = create_variation({
    "key": "gpt-5-agent",
    "name": "OpenAI GPT-5 Agent",
    "modelConfigKey": "OpenAI.gpt-5",
    "model": {"modelName": "gpt-5", "parameters": {"max_completion_tokens": 4096}},
    "messages": [],
    "instructions": (
        "You are an order triage assistant. Use the get_order_status tool to look "
        "up the order, then give a one-line status summary.\n\n"
        "Two response modes:\n"
        "1) If the user only asked about status, answer directly and STOP. Do not "
        "emit any JSON.\n"
        "2) If the user is asking for follow-up analysis (investigation, escalation, "
        "customer communication, root-cause reasoning), give the one-line status, "
        "then hand off by ending your response with this JSON envelope on its own "
        "line: `{\"route\": \"<route_name>\"}`. The available route names will be "
        "listed in your context. Keep your own response brief — the specialist "
        "produces the analysis."
    ),
})
print(f"[{'OK' if r.status_code == 201 else 'INFO'}] Variation: gpt-5-agent")

r = create_variation({
    "key": "claude-sonnet-agent",
    "name": "Anthropic Claude Sonnet Agent",
    "modelConfigKey": "Anthropic.claude-sonnet-4-6",
    "model": {"modelName": "claude-sonnet-4-6", "parameters": {"max_tokens": 4096, "temperature": 0.7}},
    "messages": [],
    "instructions": (
        "You are an order triage assistant. Use the get_order_status tool to look "
        "up the order, then give a one-line status summary.\n\n"
        "Two response modes:\n"
        "1) If the user only asked about status, answer directly and STOP. Do not "
        "emit any JSON.\n"
        "2) If the user is asking for follow-up analysis (investigation, escalation, "
        "customer communication, root-cause reasoning), give the one-line status, "
        "then hand off by ending your response with this JSON envelope on its own "
        "line: `{\"route\": \"<route_name>\"}`. The available route names will be "
        "listed in your context. Keep your own response brief — the specialist "
        "produces the analysis."
    ),
})
print(f"[{'OK' if r.status_code == 201 else 'INFO'}] Variation: claude-sonnet-agent")

# Bedrock variation: optional. We always create it so the LD UI shows the
# three-provider story, but the variation will only serve successfully if the
# notebook environment has AWS credentials with Bedrock model access.
r = create_variation({
    "key": "bedrock-claude-agent",
    "name": "Bedrock Claude Sonnet Agent",
    "model": {"modelName": "us.anthropic.claude-sonnet-4-6", "parameters": {"max_tokens": 4096, "temperature": 0.7}},
    "messages": [],
    "instructions": (
        "You are an order triage assistant. Use the get_order_status tool to look "
        "up the order, then give a one-line status summary.\n\n"
        "Two response modes:\n"
        "1) If the user only asked about status, answer directly and STOP. Do not "
        "emit any JSON.\n"
        "2) If the user is asking for follow-up analysis (investigation, escalation, "
        "customer communication, root-cause reasoning), give the one-line status, "
        "then hand off by ending your response with this JSON envelope on its own "
        "line: `{\"route\": \"<route_name>\"}`. The available route names will be "
        "listed in your context. Keep your own response brief — the specialist "
        "produces the analysis."
    ),
})
print(f"[{'OK' if r.status_code == 201 else 'INFO'}] Variation: bedrock-claude-agent")

## 3. Attach the governed tool

`get_order_status` was created centrally by `tools.ipynb`. We attach it to the `gpt-5-agent` variation (the default) so the agent has access to it when served. Attachment is via `PATCH /ai-configs/{key}/variations/{variation-id}`.

The Strands Python SDK wires tools through the `Agent(tools=[...])` constructor from application code — LD's attachment is for governance (schema + version tracking), not dispatch.


In [ ]:
# Ensure the tool exists (tools.ipynb may or may not have run in this project)
tool_body = {
    "key": TOOL_KEY,
    "name": "Get Order Status",
    "description": "Look up the status of a customer order by order ID",
    "schema": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "The order ID to look up"}
        },
        "required": ["order_id"],
    },
}
r = requests.post(f"{BASE_URL}/projects/{PROJECT_KEY}/ai-tools", headers=HEADERS, json=tool_body)
if r.status_code == 201:
    print(f"[OK] Tool created: {TOOL_KEY}")
elif r.status_code == 409:
    print(f"[INFO] Tool already exists: {TOOL_KEY}")
else:
    print(f"[ERROR] {r.status_code}: {r.text[:200]}")

# PATCH /variations/{variation-key} — attach the governed tool to each variation.
# (Takes the variation KEY, not its _id.)
for variation_key in ("gpt-5-agent", "claude-sonnet-agent", "bedrock-claude-agent"):
    r = requests.patch(
        f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}/variations/{variation_key}",
        headers=HEADERS,
        json={"tools": [{"key": TOOL_KEY, "version": 1}]},
    )
    ok = r.status_code in (200, 204)
    print(f"[{'OK' if ok else 'ERROR'}] Attached '{TOOL_KEY}' to {variation_key}  ({r.status_code})")

## 4. Set default variation & enable targeting

Point the production fallthrough at `gpt-5-agent` so the config serves a real (enabled) variation by default. Without this the config is reachable but disabled.


In [ ]:
# Read the targeting variation IDs (different from the AI Config variation _id)
r = requests.get(
    f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}/targeting",
    headers=HEADERS,
)
tvid_map = {}
for v in r.json().get("variations", []):
    if v.get("name") == "disabled":
        continue
    key = v.get("value", {}).get("_ldMeta", {}).get("variationKey")
    if key:
        tvid_map[key] = v["_id"]

fallthrough_id = tvid_map.get("gpt-5-agent")
if fallthrough_id:
    r = requests.patch(
        f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}/targeting",
        headers=PATCH_HEADERS,
        json={
            "environmentKey": ENVIRONMENT,
            "instructions": [{
                "kind": "updateFallthroughVariationOrRollout",
                "variationId": fallthrough_id,
            }],
        },
    )
    print(f"[{'OK' if r.status_code == 200 else 'ERROR'}] Default variation: gpt-5-agent")


## 5. Define the tool handler and registry

The `@tool` decorator exposes a Python function as a callable tool for the Strands `Agent`. The handler stays in application code — LD governs the schema, the app owns execution.

`TOOL_REGISTRY` maps tool *names* (matching the LD tool keys) to the local handlers. We resolve the actual list of tools to pass to the agent at runtime, from `agent_config.model.parameters['tools']` — so detaching `get_order_status` from the variation in the LaunchDarkly UI takes effect on the next agent invocation, with no code change.

The handler also fires `tracker.track_tool_call("get_order_status")` on every invocation so LD's tool-call metrics line up with what the agent actually did. (The tracker is created later in the notebook; we look it up via the module global at call time.)

In [ ]:
from strands import tool


# Module-level reference; reassigned by `run_turn` (cell 17) to the per-turn
# tracker. The @tool body looks it up at call time, so as long as `_tracker` is
# bound before the agent invokes the tool, `track_tool_call` fires on the right
# tracker. SDK 0.18 enforces at-most-once tracking per tracker, which is why
# each turn needs a fresh `create_tracker()` rather than a single shared one.
_tracker = None


@tool
def get_order_status(order_id: str) -> str:
    """Look up the status of a customer order by order ID."""
    if _tracker is not None:
        _tracker.track_tool_call("get_order_status")
    orders = {
        "ORD-123": "Shipped — arrives Thursday",
        "ORD-456": "Processing — estimated ship date: tomorrow",
        "ORD-789": "Delivered on Monday",
    }
    return orders.get(order_id, f"No order found with ID {order_id}")


# Registry maps the LD tool *key* to the local Strands tool object.
# create_strands_model + the agent build cell below use this to resolve the
# tool list from `agent_config.model.parameters['tools']` at runtime.
TOOL_REGISTRY = {"get_order_status": get_order_status}

## 6. Initialize LaunchDarkly and build the Strands agent

`create_strands_model` is the only provider-aware code: given an `LDAIAgentConfig`, it returns the matching Strands model class. Memory, invocation, and tracking are identical across providers. Tools are pulled from `agent_config.model.parameters['tools']` and resolved against `TOOL_REGISTRY`, so the LD UI is the source of truth for what the agent can call.

In [ ]:
import asyncio
import ldclient
from ldclient import Context
from ldclient.config import Config
from ldai.client import LDAIClient
from ldai.tracker import TokenUsage
from strands import Agent
from strands.models.anthropic import AnthropicModel
from strands.models.openai import OpenAIModel
from strands.models.bedrock import BedrockModel
from strands.agent.conversation_manager.sliding_window_conversation_manager import (
    SlidingWindowConversationManager,
)

ldclient.set_config(Config(SDK_KEY))
ai_client = LDAIClient(ldclient.get())
if not ldclient.get().is_initialized():
    raise RuntimeError("LaunchDarkly SDK failed to initialize")

context = Context.builder("user-123").kind("user").name("Sandy").build()
agent_config = ai_client.agent_config(CONFIG_KEY, context)
if not agent_config.enabled:
    raise RuntimeError("Agent Config is disabled")

print(f"Provider: {agent_config.provider.name if agent_config.provider else 'n/a'}")
print(f"Model:    {agent_config.model.name}")

In [ ]:
def create_strands_model(cfg):
    """Map an LDAIAgentConfig to the matching Strands model class.

    Dispatches on `provider.name` first; for Bedrock variations (which omit
    `modelConfigKey` and so have no provider field) falls back to the standard
    Bedrock model-id prefixes (`us.`, `eu.`, `apac.`, `anthropic.`, `amazon.`,
    `meta.`).
    """
    provider = (cfg.provider.name if cfg.provider else "").lower()
    model_id = cfg.model.name
    params = dict(cfg.model.to_dict().get("parameters") or {})
    # LD surfaces attached tools via `parameters.tools`; Strands gets tools
    # through the Agent constructor, so drop it from model params.
    params.pop("tools", None)

    is_bedrock = provider == "bedrock" or model_id.startswith(
        ("us.", "eu.", "apac.", "anthropic.", "amazon.", "meta.")
    )

    if is_bedrock:
        # BedrockModel takes flat kwargs (no `params` dict): pass model_id +
        # known inference fields as kwargs, drop anything else into
        # additional_request_fields so it still reaches the Converse API.
        region = (
            params.pop("region_name", None)
            or os.environ.get("AWS_REGION")
            or "us-west-2"
        )
        known = {
            k: params.pop(k)
            for k in ("max_tokens", "temperature", "top_p", "stop_sequences")
            if k in params
        }
        if "max_tokens" not in known:
            known["max_tokens"] = 1024
        return BedrockModel(
            model_id=model_id,
            region_name=region,
            additional_request_fields=params or None,
            **known,
        )
    if provider == "anthropic":
        # AnthropicModel requires max_tokens as a kwarg, not in params.
        max_tokens = int(params.pop("max_tokens", None) or params.pop("maxTokens", None) or 1024)
        return AnthropicModel(model_id=model_id, max_tokens=max_tokens, params=params or None)
    if provider == "openai":
        # gpt-5 wants max_completion_tokens; gpt-4o wants max_tokens. Keep that
        # choice in the LD variation's parameters and pass through as-is.
        return OpenAIModel(model_id=model_id, params=params)
    raise ValueError(f"Unsupported provider for Strands: {provider!r}")


model = create_strands_model(agent_config)

# Resolve the agent's tool list from the LD variation, not from a hardcoded
# Python list. `agent_config.model.parameters['tools']` is the OpenAI-style
# function shape LD returns; we only need the names to look up the local
# handlers from TOOL_REGISTRY.
ld_tool_params = (agent_config.model.to_dict().get("parameters") or {}).get("tools") or []
ld_tool_names = [t["name"] for t in ld_tool_params]

resolved_tools = []
for name in ld_tool_names:
    handler = TOOL_REGISTRY.get(name)
    if handler is None:
        print(f"[WARN] LD attached tool '{name}' has no local handler — skipping")
        continue
    resolved_tools.append(handler)
print(f"[INFO] Tools from LD: {ld_tool_names}")

# SlidingWindowConversationManager gives the agent short-term memory — reuse
# the same Agent across invoke_async calls and it carries context across turns.
agent = Agent(
    name="order-assistant",
    model=model,
    system_prompt=agent_config.instructions,
    tools=resolved_tools,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
)
print("[OK] Strands agent ready")

## 7. Run three turns

Strands' `ConversationManager` carries context between `invoke_async` calls — no explicit thread id required. LaunchDarkly's tracker records duration, token usage, and success/error per turn.


In [ ]:
from ldai.providers.types import LDAIMetrics


def strands_metrics_extractor(result):
    """Pull token usage off a Strands AgentResult into an LDAIMetrics.

    Returned to track_metrics_of_async, which fires duration/success/tokens
    atomically. Tool calls stay tracked from inside the @tool body so we don't
    double-count.
    """
    usage = getattr(result.metrics, "accumulated_usage", {}) or {}
    input_tokens = usage.get("inputTokens", 0)
    output_tokens = usage.get("outputTokens", 0)
    total = usage.get("totalTokens", 0) or (input_tokens + output_tokens)
    return LDAIMetrics(
        success=True,
        usage=TokenUsage(input=input_tokens, output=output_tokens, total=total) if total > 0 else None,
        duration_ms=None,  # let the SDK use wall-clock elapsed
    )


async def run_turn(user_input):
    # Fresh tracker per turn (SDK 0.18+ enforces at-most-once per execution).
    # Publish to module global so the @tool body fires track_tool_call on the
    # same tracker the metrics extractor finalizes. Reinit the LDClient if a
    # prior cleanup cell closed it.
    global _tracker, ai_client, agent_config
    _ld = ldclient.get()
    # is_initialized() is a one-way latch; it stays True even after close().
    # The private _closed flag is the only reliable signal of a dead client.
    _closed = getattr(_ld, "_closed", False) or getattr(_ld, "_LDClient__closed", False)
    if (not _ld.is_initialized()) or _closed:
        ldclient.set_config(Config(SDK_KEY))
        if not ldclient.get().is_initialized():
            raise RuntimeError("LaunchDarkly SDK failed to initialize")
        ai_client = LDAIClient(ldclient.get())
        agent_config = ai_client.agent_config(CONFIG_KEY, context)
        print("[INFO] Reinitialized LaunchDarkly client (was closed or stale)")
    _tracker = agent_config.create_tracker()
    try:
        result = await _tracker.track_metrics_of_async(
            strands_metrics_extractor,
            lambda: agent.invoke_async(user_input),
        )
        print(f"> {user_input}")
        print(f"  {result.message['content'][0]['text']}\n")
    except Exception as e:
        # track_metrics_of_async records track_duration + track_error on exceptions
        # before re-raising; we just surface the failure here.
        print(f"[ERROR] {e}")


async def main():
    await run_turn("What's the status of order ORD-123?")
    await run_turn("What about ORD-456?")
    await run_turn("Summarize both orders for me.")

await main()

## 8. Wire the agent into a graph

LaunchDarkly Agent Graphs let you compose multiple AI Configs into a multi-agent system: one root config, one or more target configs, and named edges (with optional `handoff` metadata) describing the routing between them. The Python SDK exposes the result via `ai_client.agent_graph(GRAPH_KEY, context)` — your application code reads the topology and dispatches between agents accordingly.

This section creates a second AI Config — `strands-specialist-agent` — and wires `strands-agent` → `strands-specialist-agent` as a one-edge graph. The actual orchestration (when to delegate to the specialist) stays in your application code; LaunchDarkly's job is to declare the graph and serve the right variation of each node per user.

### Endpoints used
- `POST /projects/{project}/ai-configs` — create the specialist node
- `POST /ai-configs/{key}/variations` — give it a variation
- `PATCH /ai-configs/{key}/targeting` — enable + set fallthrough
- `POST /projects/{project}/agent-graphs` — create the graph with a `rootConfigKey`
- `PATCH /agent-graphs/{key}` — add edges (must include `rootConfigKey` *and* `edges`; edges sent alone are silently ignored)
- `PATCH /agent-graphs/{key}` with `instructions: [{kind: "turnTargetingOn"}]` — enable the graph in production

In [ ]:
SPECIALIST_KEY = "strands-specialist-agent"
GRAPH_KEY      = "strands-agent-graph"

# 0. Delete the graph FIRST on re-runs. While a graph references a config as
# root or as an edge target, LD blocks `DELETE /ai-configs/{key}` with 409.
# Removing the graph first lets the specialist delete-then-create cycle work.
requests.delete(f"{BASE_URL}/projects/{PROJECT_KEY}/agent-graphs/{GRAPH_KEY}", headers=HEADERS)
time.sleep(0.5)

# 1. Create the specialist AI Config (the second node in the graph).
requests.delete(f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{SPECIALIST_KEY}", headers=HEADERS)
time.sleep(0.5)
r = requests.post(
    f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs",
    headers=HEADERS,
    json={"key": SPECIALIST_KEY, "name": "Strands Specialist Agent", "mode": "agent"},
)
if r.status_code == 201:
    print(f"[OK] Specialist Config: {SPECIALIST_KEY}")
elif r.status_code == 409 or (r.status_code == 400 and "enabled" in r.text):
    print(f"[INFO] Specialist Config already exists: {SPECIALIST_KEY}")
else:
    print(f"[ERROR] {r.status_code}: {r.text[:200]}")
print(f"[INFO] https://app.launchdarkly.com/projects/{PROJECT_KEY}/ai-configs/{SPECIALIST_KEY}")

# 2. Add a single variation so the specialist serves something when invoked.
r = requests.post(
    f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{SPECIALIST_KEY}/variations",
    headers=HEADERS,
    json={
        "key": "specialist-default",
        "name": "Specialist Default",
        "modelConfigKey": "OpenAI.gpt-5",
        "model": {"modelName": "gpt-5", "parameters": {"max_completion_tokens": 4096}},
        "messages": [],
        "instructions": (
            "You are a deep-reasoning order operations specialist. The triage agent "
            "hands off to you for follow-up analysis: investigation steps, escalation "
            "paths, communication templates, root-cause reasoning. The triage agent's "
            "lookup result is included in your context — use it. Be thorough and "
            "concrete. No further handoff is needed; produce the full analysis."
        ),
    },
)
print(f"[{'OK' if r.status_code == 201 else 'INFO'}] Specialist variation: specialist-default ({r.status_code})")

# 3. Enable the specialist by pointing fallthrough at the variation we just made.
time.sleep(0.5)
r = requests.get(f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{SPECIALIST_KEY}/targeting", headers=HEADERS)
spec_vmap = {
    v.get("value", {}).get("_ldMeta", {}).get("variationKey"): v["_id"]
    for v in r.json().get("variations", [])
    if v.get("name") != "disabled"
}
spec_id = spec_vmap.get("specialist-default")
if spec_id:
    r = requests.patch(
        f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{SPECIALIST_KEY}/targeting",
        headers=PATCH_HEADERS,
        json={
            "environmentKey": ENVIRONMENT,
            "instructions": [{"kind": "updateFallthroughVariationOrRollout", "variationId": spec_id}],
        },
    )
    print(f"[{'OK' if r.status_code == 200 else 'INFO'}] Specialist fallthrough set ({r.status_code})")

# 4. Create the Agent Graph with strands-agent as the root.
r = requests.post(
    f"{BASE_URL}/projects/{PROJECT_KEY}/agent-graphs",
    headers=HEADERS,
    json={
        "key": GRAPH_KEY,
        "name": "Strands Agent Graph",
        "description": "Root strands-agent with a deep-reasoning specialist.",
        "rootConfigKey": CONFIG_KEY,
    },
)
print(f"[{'OK' if r.status_code == 201 else 'INFO'}] Agent Graph: {GRAPH_KEY} ({r.status_code})")

# 5. Add the edge. NOTE: the API requires `rootConfigKey` to be re-sent with
# `edges`; edges sent alone are silently ignored.
r = requests.patch(
    f"{BASE_URL}/projects/{PROJECT_KEY}/agent-graphs/{GRAPH_KEY}",
    headers=HEADERS,
    json={
        "rootConfigKey": CONFIG_KEY,
        "edges": [{
            "key": "edge-strands-specialist",
            "sourceConfig": CONFIG_KEY,
            "targetConfig": SPECIALIST_KEY,
            "handoff": {"route": "specialist", "reason": "needs deep reasoning"},
        }],
    },
)
print(f"[{'OK' if r.status_code == 200 else 'ERROR'}] Edge added: {CONFIG_KEY} → {SPECIALIST_KEY}")

# 6. Turn the graph on in production. Without this the SDK reads it back as disabled.
r = requests.patch(
    f"{BASE_URL}/projects/{PROJECT_KEY}/agent-graphs/{GRAPH_KEY}",
    headers=HEADERS,
    json={"instructions": [{"kind": "turnTargetingOn"}]},
)
print(f"[{'OK' if r.status_code == 200 else 'ERROR'}] Graph enabled in {ENVIRONMENT}")
print(f"[INFO] https://app.launchdarkly.com/projects/{PROJECT_KEY}/ai/graphs?env={ENVIRONMENT}&selected-env={ENVIRONMENT}")

# 7. Read it back via the SDK and print the topology.
graph = ai_client.agent_graph(GRAPH_KEY, context)
print(f"\nGraph enabled: {graph.is_enabled()}")
root = graph.root()
if root:
    print(f"Root: {root.get_key()}")
    for edge in root.get_edges():
        print(f"  → {edge.target_config}  (handoff: {edge.handoff})")
print(
    "\n[INFO] Application code reads `graph.root()` / `graph.get_child_nodes(...)` "
    "and dispatches between agents itself; LaunchDarkly only declares the topology."
)

## 9. Graph-driven dispatcher

Section 7 invokes the root agent directly; section 8 only *declares* the graph topology. To make the LaunchDarkly graph actually drive Strands behavior — so adding a node + edge in the LaunchDarkly UI changes which agents run, with no code change — fetch the graph at runtime and traverse it.

Three pieces, all parameterized on whatever the graph contains:

1. **`build_strands_agent(node_key, config, valid_routes)`** — generic Strands `Agent` builder. Reads model class, instructions, parameters, and the tool list from the AI Config. If the node has outgoing edges, appends a routing instruction so the LLM emits `{"route": "..."}` at the end of its response.
2. **`extract_route(text, valid_routes)`** — parses that JSON and validates against the edges' `handoff.route` values.
3. **`execute_graph(graph_key, user_input, context)`** — walks the graph: at each node, build the agent, invoke it with the previous node's output, parse the route, jump to the matching edge target. Terminates at the first node with no outgoing edges. Tracks per-node metrics with `config.create_tracker()` (duration / tokens / success+error / tool calls) **and** graph-level handoffs + final path with `graph.create_tracker()`.

`GRAPH_KEY` is the only key the notebook hardcodes from here on — every agent is materialized from whatever the graph contains at runtime.

In [ ]:
import json
import re


def build_strands_agent(node_key, config, edges):
    """Build a Strands Agent generically from any AIAgentConfig.

    Reads model, instructions, parameters, and attached tools from the config.
    The routing protocol (JSON envelope, when to hand off) lives in each AI
    Config's `instructions` field in LaunchDarkly — this builder only surfaces
    the runtime-derived *list* of available routes from the graph's edges, so
    that adding/removing an edge in LD changes what the LLM sees, with no code
    change.
    """
    model = create_strands_model(config)
    instructions = config.instructions or "Process the input and respond helpfully."

    # Build the runtime route list from edges (LD-stored handoff metadata).
    # `route` is the value the LLM emits; `reason` is the editable description.
    route_lines = []
    for e in edges:
        if not e.handoff:
            continue
        route = e.handoff.get("route")
        if not route:
            continue
        reason = e.handoff.get("reason")
        route_lines.append(f"- `{route}` — {reason}" if reason else f"- `{route}`")

    if route_lines:
        instructions += "\n\nAvailable routes:\n" + "\n".join(route_lines)

    # Tool list from LD, resolved against the local TOOL_REGISTRY.
    ld_tool_params = (config.model.to_dict().get("parameters") or {}).get("tools") or []
    tool_names = [t["name"] for t in ld_tool_params]
    resolved_tools = [TOOL_REGISTRY[n] for n in tool_names if n in TOOL_REGISTRY]
    missing = [n for n in tool_names if n not in TOOL_REGISTRY]
    if missing:
        print(f"[WARN] {node_key}: LD attached tools {missing} have no local handler")

    return Agent(
        name=node_key,
        model=model,
        system_prompt=instructions,
        tools=resolved_tools,
        conversation_manager=SlidingWindowConversationManager(window_size=40),
        # Suppress Strands' default stdout streaming so the dispatcher prints
        # each agent's response exactly once, under its banner.
        callback_handler=None,
    )


def extract_route(text, valid_routes):
    """Find a {"route": "..."} object in the LLM response and validate it."""
    if not valid_routes:
        return None
    patterns = [
        r'```json\s*(\{[^`]+\})\s*```',
        r'(\{"route":\s*"[^"]+"\s*\})',
        r'(\{[^{}]*"route"\s*:\s*"[^"]+"[^{}]*\})',
    ]
    valid_lower = [r.lower() for r in valid_routes]
    for pat in patterns:
        m = re.search(pat, text, re.DOTALL)
        if not m:
            continue
        try:
            data = json.loads(m.group(1))
        except json.JSONDecodeError:
            continue
        route = (data.get("route") or "").lower().strip()
        if route in valid_lower:
            return valid_routes[valid_lower.index(route)]
    return None

In [ ]:
async def execute_graph(graph_key, user_input, context):
    """Walk the LaunchDarkly Agent Graph, invoking each node's Strands agent.

    No hardcoded agent keys. Routing decisions come from each agent's response
    (a JSON route), validated against the node's outgoing edges. If a node has
    edges but the agent does not emit a route, the dispatcher terminates at that
    node — handoffs are optional, governed by the agent's instructions.

    Self-heals across re-runs: if the LDClient was closed by a prior cleanup
    cell (or never initialized in this kernel), reinitialize it + rebuild the
    LDAIClient wrapper before fetching the graph. Once initialized, the SDK's
    streaming connection keeps state fresh within ~1s of any LD UI change.
    """
    global _tracker, ai_client
    _ld = ldclient.get()
    # is_initialized() is a one-way latch; it stays True even after close().
    # The private _closed flag is the only reliable signal of a dead client.
    _closed = getattr(_ld, "_closed", False) or getattr(_ld, "_LDClient__closed", False)
    if (not _ld.is_initialized()) or _closed:
        ldclient.set_config(Config(SDK_KEY))
        if not ldclient.get().is_initialized():
            raise RuntimeError("LaunchDarkly SDK failed to initialize")
        ai_client = LDAIClient(ldclient.get())
        print("[INFO] Reinitialized LaunchDarkly client (was closed or stale)")
    graph = ai_client.agent_graph(graph_key, context)
    if not graph.is_enabled():
        raise RuntimeError(f"Agent Graph '{graph_key}' is not enabled")

    graph_tracker = graph.create_tracker()

    nodes = {}
    graph.reverse_traverse(lambda n, _: nodes.update({n.get_key(): n}), {})
    print(f"[INFO] Graph '{graph_key}' has {len(nodes)} node(s): {list(nodes.keys())}")

    current = graph.root()
    visited = set()
    path = []
    current_input = user_input
    prev_key = None

    try:
        while current is not None:
            key = current.get_key()
            if key in visited:
                raise RuntimeError(f"Cycle detected at node: {key}")
            visited.add(key)
            path.append(key)

            config = current.get_config()
            edges = current.get_edges()
            valid_routes = [e.handoff["route"] for e in edges if e.handoff and e.handoff.get("route")]

            if prev_key is not None:
                graph_tracker.track_handoff_success(prev_key, key)

            agent = build_strands_agent(key, config, edges)
            print(f"\n┌─ INVOKED agent: {key} ─────────────────────────")
            # Show the input this agent receives. For the root that's the
            # user's original query; for downstream agents it's the prior
            # agent's full response (the conversation hop).
            preview = current_input if len(current_input) <= 600 else current_input[:600] + "..."
            print(f"│ input ({len(current_input)} chars):")
            for line in preview.splitlines() or [""]:
                print(f"│   {line}")
            print("├─ response ─")

            _tracker = config.create_tracker()
            try:
                result = await _tracker.track_metrics_of_async(
                    strands_metrics_extractor,
                    lambda: agent.invoke_async(current_input),
                )
            except Exception as e:
                print(f"└─ [ERROR] {key}: {e}")
                raise

            response_text = result.message["content"][0]["text"]
            print(response_text)
            print(f"└─ done: {key}")

            if not edges:
                break

            route = extract_route(response_text, valid_routes)
            if route is None:
                # Agent omitted the JSON envelope — that's the "stop here" signal.
                # No handoff attempted, no failure event.
                print(f"[INFO] {key} omitted route JSON — terminating here.")
                break

            next_key = None
            for edge in edges:
                if edge.handoff and edge.handoff.get("route") == route:
                    next_key = edge.target_config
                    break
            if next_key is None:
                # Emitted an unrecognized route — record a real handoff failure
                # against the intended target (first edge) and stop.
                target_guess = edges[0].target_config
                graph_tracker.track_handoff_failure(key, target_guess)
                print(f"[INFO] {key} emitted unrecognized route '{route}'; stopping.")
                break

            print(f"[INFO] {key} chose route '{route}' → {next_key}")
            prev_key = key
            current = nodes.get(next_key)
            current_input = response_text  # downstream agent sees prior agent's output

        graph_tracker.track_path(path)
        graph_tracker.track_invocation_success()
        print(f"\n[OK] Path invoked: {' → '.join(path)}")
    except Exception:
        graph_tracker.track_invocation_failure()
        raise


# Two queries demonstrating both branches:
#  - Simple status lookup: triage answers, no handoff (only strands-agent fires)
#  - Complex follow-up:    triage hands off to specialist (both fire)

print("\n========== Query 1: status only ==========")
await execute_graph(GRAPH_KEY, "What's the status of order ORD-789?", context)

print("\n========== Query 2: needs analysis ==========")
await execute_graph(
    GRAPH_KEY,
    "Order ORD-456 has been stuck in 'Processing' for 5 days. The customer is asking why. "
    "Walk me through how to investigate and what to say to the customer.",
    context,
)

In [ ]:
# Flush buffered events so they reach LaunchDarkly. We intentionally do NOT
# call ldclient.get().close() here — close() puts the singleton into a state
# where subsequent ai_client.agent_graph(...) calls hit a cached snapshot
# (and emit "evaluation attempted before client has initialized" warnings)
# instead of the live LD state. Kernel shutdown releases the connection.
ldclient.get().flush()
print("[OK] Done")


In [ ]:
# Uncomment to delete resources
# requests.delete(f"{BASE_URL}/projects/{PROJECT_KEY}/ai-configs/{CONFIG_KEY}", headers=HEADERS)
# requests.delete(f"{BASE_URL}/projects/{PROJECT_KEY}", headers=HEADERS)

## What this sample showed

- **One agent config, three providers.** `create_strands_model` picks the Strands model class from `config.provider.name` (with a Bedrock model-id fallback). No per-provider branching in your code.
- **Tools driven by LaunchDarkly.** Each agent's tool list comes from `config.model.parameters['tools']`, resolved against a local `TOOL_REGISTRY`. Detach `get_order_status` from a variation in the LaunchDarkly UI and the next invocation has no tools — no code change.
- **Governed tools.** `get_order_status` is registered centrally via `POST /ai-tools` and attached per-variation via `PATCH /ai-configs/.../variations/{id}`. The Python handler stays local; LaunchDarkly tracks the schema + version.
- **Multi-turn memory.** Strands' `SlidingWindowConversationManager` plus a reused `Agent` instance — no thread IDs, no explicit state store. Section 7 demonstrates this on the root agent.
- **Per-config tracking.** `config.create_tracker().track_duration_of` + `track_success/error` + `track_tokens`, plus `track_tool_call` from inside the `@tool` body, feed the AI Config's Monitoring tab.
- **Graph-driven multi-agent dispatch.** Section 9's `execute_graph` is the load-bearing piece: it reads the topology from LaunchDarkly at runtime, builds a Strands `Agent` per node generically from `node.get_config()`, and routes between agents based on `edge.handoff.route` values that each LLM selects from the valid set. The graph tracker records handoffs + the final path.
- **The only key in the runtime path is `GRAPH_KEY`.** Add a node + edge in the LaunchDarkly UI, re-run section 9, and the new agent participates — no code change.

## Try it

- **Swap providers.** Set the default variation on `strands-agent` to `claude-sonnet-agent` or `bedrock-claude-agent` and re-run section 9 — same agent code, different provider.
- **Add an agent.** Create a third AI Config (e.g. `strands-fraud-agent`), add it as a node in the `strands-agent-graph`, and add an edge from `strands-agent` with `handoff: {route: "fraud"}`. Re-run section 9 with a fraud-flavored query and watch the new node light up in the Agent Graph view.
- **Detach a tool.** Detach `get_order_status` from the served variation in the UI and re-run — the root agent stops calling it.

## Additional resources
- [LaunchDarkly AI Configs documentation](https://launchdarkly.com/docs/home/ai-configs)
- [LaunchDarkly Agent Graphs](https://launchdarkly.com/docs/home/ai-configs/agent-graphs)
- [LaunchDarkly AI SDK for Python](https://github.com/launchdarkly/python-server-sdk-ai)
- [Strands Agents SDK](https://github.com/strands-agents/sdk-python)